# Fit Model parameters to single variable data (voltage)

This notebook follows the [PBPram example of the same name](https://github.com/paramm-team/pybamm-param/blob/develop/examples/notebooks/datafit_single_variable.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybop
import pybamm

# 1. Load the data
The first step is to load the data to which we want to fit the model. For this example, we use the experimental dataset which from [Brosa Planella et al. (2021) article](https://www.sciencedirect.com/science/article/pii/S0013468621008148). This data set is already in the right format, if you are using your own dataset you should ensure that the headers of the relevant columns match the variable names in PyBaMM (e.g. `"Time [s]"`, `"Voltage [V]"`...)

In [12]:
# Data is up two directory then in data/pybamm/
data_path = Path.cwd().parent / "data" / "pybamm"/"LGM50_789_1C_25degC.csv"
data = pd.read_csv(data_path)

# This data has some issues in the time column, so we will fix it
mask = data["Time [s]"].values[:-1] < data["Time [s]"].values[1:]  # Check if the time is increasing

# where the mask is false, we will drop the row
data = data.iloc[:-1][mask]


# Transform the data into a pybop dataset
dataset = pybop.Dataset(
    {
     "Time [s]": data["Time [s]"].values,
     "Voltage [V]": data["Voltage [V]"].values,
     "X-averaged cell temperature [K]": data["X-averaged cell temperature [K]"].values,
     "Current function [A]": np.ones_like(data["Voltage [V]"].values)
    }
)


# 2. Define the model
Next we need to define the model we want to fit to the data. This also includes defining the solver, spatial methods, parameters that we are not fitting, and operating conditions. To streamline this, we wrap everything into a PyBaMM simulation. The basic idea is that the simulation already includes everything needed to solve the model under certain conditions.

In this case we choose the Single Particle Model (SPM) with a contact resistance, which we will fit to the data.

In [13]:
# Define the model and parameter set
parameter_set = pybop.ParameterSet.pybamm("Chen2020")

model = pybop.lithium_ion.SPM(parameter_set=parameter_set, options={"contact resistance": "true"})
# Define the operating conditions
experiment = pybamm.Experiment(
    [
        "Discharge at 1C until 2.5 V",
        "Rest for 2 hours",
    ],
    period="30 seconds",
)

# 3. Define the Optimisation Problem 

In [14]:
# Define the optimization parameters with initial values and bounds

parameters = pybop.Parameters(
    pybop.Parameter(
        "Negative particle diffusivity [m2.s-1]",
        initial_value=5e-14,
        bounds=(2.06e-16, 2.06e-12),
    ),
    pybop.Parameter(
        "Contact resistance [Ohm]",
        initial_value=0,
        bounds=(0, 0.5),
    ),
) 

In [15]:
for i in parameters.keys():
    print(i)
    print(f"{i}: {model.parameter_set[i]}")

Negative particle diffusivity [m2.s-1]
Negative particle diffusivity [m2.s-1]: 3.3e-14
Contact resistance [Ohm]
Contact resistance [Ohm]: 0


In [16]:
solution = model.predict(experiment=experiment)

In [17]:
problem = pybop.FittingProblem(model=model, parameters=parameters, dataset=dataset)


In [18]:
cost = pybop.SumSquaredError(problem)

In [19]:
optim = pybop.SciPyMinimize(
    cost,
    method="Nelder-Mead",
)

In [20]:
results = optim.run()

OptimisationResult:
  Initial parameters: [5.e-14 0.e+00]
  Optimised parameters: [3.82940555e-16 5.00000000e-01]
  Diagonal Fisher Information entries: None
  Final cost: 677.2748468514975
  Optimisation time: 13.895312786102295 seconds
  Number of iterations: 62
  Number of evaluations: None
  SciPy result available: Yes


In [21]:
pybop.plot.quick(problem=problem, problem_inputs=results.x)

[Figure({
     'data': [{'fill': 'toself',
               'fillcolor': 'rgba(255,229,204,0.8)',
               'hoverinfo': 'skip',
               'line': {'color': 'rgba(255,255,255,0)'},
               'showlegend': False,
               'type': 'scatter',
               'x': [0.0, 0.0010000000002037, 0.5079999999998108, ...,
                     0.5079999999998108, 0.0010000000002037, 0.0],
               'y': [3.85363160477957, 3.8536306663567417, 3.853164698184845, ...,
                     3.4320763630406392, 3.432542331212536, 3.432543269635364]},
              {'mode': 'markers',
               'name': 'Reference',
               'showlegend': True,
               'type': 'scatter',
               'x': array([0.0000000e+00, 1.0000000e-03, 5.0800000e-01, ..., 1.0641600e+04,
                           1.0642506e+04, 1.0643563e+04]),
               'y': array([4.17955, 4.02629, 4.01529, ..., 3.11854, 3.11854, 3.11854])},
              {'mode': 'lines',
               'name': 'Mode